# SPX Trend

## Contents

- [Configuration](#configuration)
- [Market Data](#market-data)
- [Trend Indicators](#trend-indicators)
- [Chart](#chart)
- [Slide](#slide)
- [Video Script](#video-script)
- [Video](#video)

## Configuration
[top](#contents)

In [1]:
support_resistance_levels = [
    {"idx": 1, "date": "2025-07-03", "value": 6284.65, type: "resistance"},
    {"idx": 2, "date": "2025-02-19", "value": 6147.43, type: "support"},
    {"idx": 3, "date": "2025-06-11", "value": 6059.40, type: "support"},
    {"idx": 4, "date": "2025-05-19", "value": 5968.61, type: "support"},
    {"idx": 5, "date": "2025-03-25", "value": 5786.95, type: "support"},
]

In [3]:
from pydantic import BaseModel
from typing import Optional
from decimal import Decimal


class SpxTechnical(BaseModel):
    name: str
    trading_day: str
    interval: str
    high: str
    low: str
    close: str
    change_percent: str
    change_text: str
    volume: str
    volume_text: str
    volume_interpretation: str
    median_1year_volume: str
    supertrend_bias: str
    support_level: str
    support_date: str
    resistance_level: Optional[str] = None
    resistance_date: Optional[str] = None

## Market Data
[top](#contents)

In [11]:
import yfinance as yf
import pandas as pd


def get_spx_data(start: str, end: str) -> pd.DataFrame:
    # Define the ticker symbol for the S&P 500 index
    ticker = "^SPX"

    # Download historical data
    df = yf.download(ticker, start=start, end=end, interval="1d", auto_adjust=True)

    # Flatten multi-index columns
    if isinstance(df.columns, pd.MultiIndex):
        column_names = ["Date"] + df.columns.get_level_values(0).tolist()
        df = df.reset_index()
        df.columns = column_names
        df = df.set_index("Date")

    # change prices to integer
    df[["Open", "High", "Low", "Close"]] = df[["Open", "High", "Low", "Close"]].round(2)

    # Save to CSV
    df.to_csv("data/spx.csv")
    print("Data saved to data/spx.csv")

    return df

**Download SPX data.**

In [12]:
from pulse.utils.date import get_start_and_end
from pulse.utils.data import compute_percent_change

start, end = get_start_and_end()
df = get_spx_data(start, end)
compute_percent_change(df)
df.tail()

[*********************100%***********************]  1 of 1 completed

Data saved to data/spx.csv


,Close,High,Low,Open,Volume,Change%
Date,,,,,,
2026-03-26,6477.16,6573.22,6473.79,6555.86,4845560000,-1.740621
2026-03-27,6368.85,6453.89,6356.08,6453.89,5303490000,-1.672183
2026-03-30,6343.72,6427.31,6316.91,6403.37,5458640000,-0.394577
2026-03-31,6528.52,6539.05,6395.88,6395.88,6396100000,2.913117
2026-04-01,6575.32,6609.67,6554.29,6556.56,5637260000,0.716855


In [15]:
import yfinance as yf
import pandas as pd

# 2. Resample to Weekly, setting Monday as the start and Friday as the end
# 'W-FRI' means weekly, ending on Friday.
weekly_data = df.resample('W-FRI').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
})

print(weekly_data.head())
print(weekly_data.tail())


               Open     High      Low    Close       Volume
Date                                                       
2025-04-04  5580.76  5695.31  5069.90  5074.08  20307800000
2025-04-11  4953.79  5481.34  4835.04  5363.36  37869410000
2025-04-18  5441.96  5459.46  5220.79  5282.70  18671180000
2025-04-25  5232.94  5528.11  5101.63  5525.21  23198970000
2025-05-02  5529.22  5700.70  5433.24  5686.67  24244170000
               Open     High      Low    Close       Volume
Date                                                       
2026-03-06  6824.36  6901.01  6710.42  6740.02  29555750000
2026-03-13  6699.80  6845.08  6623.92  6632.19  29794740000
2026-03-20  6674.37  6754.30  6473.52  6506.48  31514370000
2026-03-27  6574.96  6651.62  6356.08  6368.85  26385000000
2026-04-03  6403.37  6609.67  6316.91  6575.32  17492000000


## Trend Indicators
[top](#contents)

In [ ]:
import numpy as np


def supertrend_indicator(df, period=10, multiplier=3):
    # Calculate True Range (TR)
    hl = df["High"] - df["Low"]
    hc = np.abs(df["High"] - df["Close"].shift())
    lc = np.abs(df["Low"] - df["Close"].shift())
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)

    # Average True Range (ATR)
    atr = tr.rolling(period).mean()

    # Middle price
    hl2 = (df["High"] + df["Low"]) / 2

    # Upper and Lower Bands
    upperband = hl2 + (multiplier * atr)
    lowerband = hl2 - (multiplier * atr)

    # Initialize Supertrend
    dir_ = [1] * len(df)
    trend = [np.nan] * len(df)
    up = [np.nan] * len(df)
    down = [np.nan] * len(df)

    for i in range(1, len(df)):
        if df["Close"].iloc[i] > upperband.iloc[i - 1]:
            dir_[i] = 1
        elif df["Close"].iloc[i] < lowerband.iloc[i - 1]:
            dir_[i] = -1
        else:
            dir_[i] = dir_[i - 1]
            if dir_[i] > 0 and lowerband.iloc[i] < lowerband.iloc[i - 1]:
                lowerband.iloc[i] = lowerband.iloc[i - 1]
            if dir_[i] < 0 and upperband.iloc[i] > upperband.iloc[i - 1]:
                upperband.iloc[i] = upperband.iloc[i - 1]

        if dir_[i] > 0:
            trend[i] = up[i] = lowerband.iloc[i]
        else:
            trend[i] = down[i] = upperband.iloc[i]

    df["Supertrend"] = pd.Series(trend, index=df.index)
    df["Trend"] = pd.Series(dir_, index=df.index)
    df["Up"] = pd.Series(up, index=df.index)
    df["Down"] = pd.Series(down, index=df.index)

    return df

In [ ]:
def support_resistance_line(df, date, price, extension=5):

    # Find the index
    pivot_index = df.index.get_loc(date)

    # Create a Series for the horizontal line
    line = pd.Series([np.nan] * len(df), index=df.index)

    # Extend the line to the right and 5 days to the left
    left_index = max(0, pivot_index - extension)
    right_index = len(df) - 1  # Extend to the end of the DataFrame
    line.iloc[left_index : right_index + 1] = price

    return line

**Add Supertrend data.**

In [ ]:
df = supertrend_indicator(df)
up_trend = df[["Up"]]
down_trend = df[["Down"]]

**Add support & resistance data.**

## Chart
[top](#contents)

In [ ]:
import mplfinance as mpf
import matplotlib.pyplot as plt


def create_chart(df):

    # Indicators
    sr_levels = support_resistance_levels
    sr_item_1 = next(item for item in sr_levels if item["idx"] == 1)
    sr_item_2 = next(item for item in sr_levels if item["idx"] == 2)
    sr_item_3 = next(item for item in sr_levels if item["idx"] == 3)

    sr1 = support_resistance_line(df, sr_item_1["date"], sr_item_1["value"])
    sr2 = support_resistance_line(df, sr_item_2["date"], sr_item_2["value"])
    sr3 = support_resistance_line(df, sr_item_3["date"], sr_item_3["value"])

    curve_sr_1 = mpf.make_addplot(sr1, color="blue", width=1.0, linestyle="--")
    curve_sr_2 = mpf.make_addplot(sr2, color="blue", width=1.0, linestyle="--")
    curve_sr_3 = mpf.make_addplot(sr3, color="blue", width=1.0, linestyle="--")

    curve_trend_up = mpf.make_addplot(
        up_trend,
        color="green",
        width=0.5,
    )

    curve_trend_down = mpf.make_addplot(
        down_trend,
        color="#FF8849",
        width=0.5,
    )

    addplot = [
        curve_sr_1,
        curve_sr_2,
        curve_sr_3,
        curve_trend_up,
        curve_trend_down,
    ]

    # Fill Between Method
    fill_trend_up = dict(y1=df["Up"].values, y2=df["Low"].values, alpha=0.05, color="g")
    fill_trend_down = dict(
        y1=df["Down"].values, y2=df["High"].values, alpha=0.05, color="r"
    )

    fill_between = [
        fill_trend_up,
        fill_trend_down,
    ]

    # Save chart with 16:9 aspect ratio and 1080p resolution
    # figsize = (16, 9)          # width, height in inches
    # dpi = 120                  # 16*120 = 1920, 9*120 = 1080
    # figsize = (1920/dpi, 1080/dpi)

    mpf.plot(
        df,
        type="ohlc",
        addplot=addplot,
        fill_between=fill_between,
        volume=True,
        style="charles",
        figsize=(16, 9),
        tight_layout=True,
        xrotation=0,
        datetime_format="%Y-%m-%d",
        savefig="../output/data/spx/chart.svg",
        # figsize=(19.2, 10.8),
        # tight_layout=False,
        # scale_padding=0.8,
        # savefig=dict(fname="../output/data/spx/chart.png", dpi=100),
    )

    # Display chart.
    # mpf.plot(
    #     df,
    #     type="ohlc",
    #     addplot=addplot,
    #     fill_between=fill_between,
    #     volume=True,
    #     style="charles",
    #     figratio=(16, 9),
    #     tight_layout=True,
    #     xrotation=0,
    #     datetime_format="%Y-%m-%d",
    # )

In [ ]:
create_chart(df)

## Video Script
[top](#contents)

In [ ]:
def spx_video_script_en(c: SpxTechnical) -> str:
    script = f"""
As of the last trading session on {c.trading_day}, the S&P 500 closed at {c.close} points, with a daily change of {c.spx_change_percent}% compared to the previous close.
Trading volume for the session was {c.volume} units. This may indicate {c.volume_interpretation}.
Based on the Supertrend indicator, the current market is {c.supertrend_bias} bias.
Looking at technical levels:
- The nearest **support** is at {c.support_level} — this was formed from a recent pivot low on {c.support_date}.
- The nearest **resistance** is at {c.resistance_level}, established from a pivot high on {c.resistance_date}.
"""
    return script.strip()

In [ ]:
def spx_video_script_zh(c: SpxTechnical) -> str:
    if c.resistance_level is None:
        resistance_text = "而阻力位尚未形成。"
    else:
        resistance_text = f"而阻力位则在{c.resistance_level}点。"

    script = f"""
截至上一个交易日{c.trading_day}，标普500指数收于{c.close}点，{c.change_text}。
当日的成交量为{c.volume_text}，{c.volume_interpretation}。
根据'超级趋势'指标，目前市场呈现{c.supertrend_bias}走势。
当前的重要支撑位在{c.support_level}点；{resistance_text}
"""
    return script.strip()

案例一：明显高于一年平均值
今天的成交量显著放大，远高于过去一年的平均水平，显示市场情绪异常活跃，资金进出频繁，可能意味着重要消息面驱动或关键趋势正在酝酿中。

案例二：高于一年平均值
今天的成交量高于一年平均水平，表明市场参与度提升，买卖力量增强，值得关注价格是否随之形成突破或延续既有趋势。

案例三：明显低于一年平均值
今天的成交量明显萎缩，远低于过去一年的平均值，反映市场观望情绪浓厚，主力资金多在场外观望，短期可能缺乏明确方向。

案例四：低于一年平均值
今天的成交量略低于一年平均水平，显示交易相对清淡，市场动力减弱，投资人可能在等待新的催化因素出现。

案例五：与一年平均值大致相当
今天的成交量与过去一年的平均水平相近，说明市场维持正常活跃度，当前走势暂未出现明显放量或缩量信号。

In [ ]:
def get_trend_zh(trend: int) -> str:
    if trend == 1:
        return "看涨"
    elif trend == -1:
        return "看跌"
    else:
        return trend

In [ ]:
def get_spx_change_text(change_percent):
    if change_percent > 0:
        text = f"较前一日上涨了{change_percent:.2f}% "
    elif change_percent < 0:
        text = f"较前一日下跌了{abs(change_percent):.2f}% "
    else:
        text = ""
    return text

In [ ]:
def get_volume_interpretation(df, volume):
    median = df["Volume"].median()
    if volume >= 1.5 * median:
        text = "远高于过去一年的平均水平，显示市场情绪异常活跃，资金进出频繁"
    elif 1.2 * median <= volume < 1.5 * median:
        text = "高于一年平均水平"
    elif 0.8 * median <= volume < 1.2 * median:
        text = "与过去一年的平均水平相近，说明市场维持正常活跃度"
    elif 0.5 * median <= volume < 0.8 * median:
        text = "略低于一年平均水平"
    elif volume < 0.5 * median:
        text = "远低于过去一年的平均值，反映市场观望情绪浓厚"
    else:
        text = ""
    return text

In [ ]:
def num_to_chinese(num):
    units = ["", "万", "亿", "兆"]
    unit_thresholds = [1, 10_000, 100_000_000, 1_0000_0000_0000]

    # Find the largest applicable unit
    for i in range(len(unit_thresholds) - 1, -1, -1):
        if num >= unit_thresholds[i]:
            value = num / unit_thresholds[i]
            # Format to 2 decimal places, remove .0 if integer
            if value.is_integer():
                return f"{int(value)}{units[i]}"
            else:
                return f"{value:.2f}{units[i]}"
    return str(num)

In [ ]:
# last = df.loc["2025-06-20"]
last = df.iloc[-1]
change_pct = last["Change%"]

data = {
    "name": "标普500指数",
    "trading_day": last.name.strftime("%Y-%m-%d"),
    "interval": "1d",
    "high": "{:.2f}".format(last["High"]),
    "low": "{:.2f}".format(last["Low"]),
    "close": "{:.2f}".format(last["Close"]),
    "change_percent": "{:.2f}%".format(last["Change%"]),
    "change_text": get_spx_change_text(change_pct),
    "volume": "{:.0f}".format(last["Volume"]),
    "volume_text": num_to_chinese(last["Volume"]),
    "volume_interpretation": get_volume_interpretation(df, last["Volume"]),
    "median_1year_volume": "{:.0f}".format(df["Volume"].median()),
    "supertrend_bias": get_trend_zh(last["Trend"]),
    "support_level": "6147.43",
    "support_date": "2025-02-19",
    "resistance_level": "6284.65",
    "resistance_date": "2025-07-03",
}

In [ ]:
data

In [ ]:
from pulse.utils.script import save_script, save_technical

c = SpxTechnical(**data)
script = spx_video_script_zh(c)

filejson = "../output/data/spx/technical.json"
save_technical(c, filejson)

filename = "../output/data/spx/script.txt"
save_script(script, filename)

print(script)